In [1]:
import os
import pandas as pd

In [2]:
import os
import pandas as pd

mimic_iv_cxr_parent = "/mnt/nfs_share/Public_Data/Dataset_MIMIC_CXR_JPG/physionet.org/files/mimic-cxr-jpg/2.0.0"

mm_dir = "/mnt/data/yihua/master/datasets/mimic-iv"
preprocessing_dir = os.path.join(mm_dir, "preprocessing")
f_path = os.path.join(preprocessing_dir, "cxr_embeddings.pkl")

In [3]:
f_path = os.path.join(mimic_iv_cxr_parent, "mimic-cxr-2.0.0-metadata.csv")
meta_data_df = pd.read_csv(f_path, low_memory=False)

f_path = os.path.join(preprocessing_dir, "cxr_embeddings.pkl")
cxr_embeddings_df = pd.read_pickle(f_path)

clean_time = meta_data_df['StudyTime'].astype(str).str.split('.').str[0].str.zfill(6)
clean_date = meta_data_df['StudyDate'].astype(str)

# 產生計算好的 datetime 欄位
meta_data_df['cxrtime'] = pd.to_datetime(
    clean_date + clean_time, 
    format='%Y%m%d%H%M%S', 
    errors='coerce'
)

cxr_embeddings_df = cxr_embeddings_df.merge(
    meta_data_df[['dicom_id', 'cxrtime']], 
    on='dicom_id', 
    how='left'
)

In [4]:
mimic_iv_path = "/mnt/nfs_share/Public_Data/Dataset_MIMICs/physionet.org/files/mimic-iv/2.2/"
icustays_df = pd.read_csv(os.path.join(mimic_iv_path, "icu", "icustays.csv"), low_memory=False)
icustays_df['intime'] = pd.to_datetime(icustays_df['intime'])
icustays_df['outtime'] = pd.to_datetime(icustays_df['outtime'])

admissions_df = pd.read_csv(os.path.join(mimic_iv_path, "hosp", "admissions.csv"), low_memory=False)
admissions_df['admittime'] = pd.to_datetime(admissions_df['admittime'])
admissions_df['dischtime'] = pd.to_datetime(admissions_df['dischtime'])

In [6]:
from tqdm import tqdm

def calc_time_delta_hrs(icu_intime, charttime):
    return (charttime - icu_intime).total_seconds() / 3600

cxr_embeddings_df['hadm_id'] = None
cxr_embeddings_df['stay_id'] = None
cxr_embeddings_df['icu_time_delta'] = None
cxr_embeddings_df['hosp_time_delta'] = None

for index, row in tqdm(cxr_embeddings_df.iterrows(), total=cxr_embeddings_df.shape[0]):
    curr_pts_icustays = icustays_df[icustays_df['subject_id'] == row['subject_id']]
    
    for icu_index, icu_row in curr_pts_icustays.iterrows():
        if icu_row['intime'] <= row['cxrtime'] <= icu_row['outtime']:
            cxr_embeddings_df.loc[index, 'stay_id'] = icu_row['stay_id']
            cxr_embeddings_df.loc[index, 'icu_time_delta'] = calc_time_delta_hrs(icu_row['intime'], row['cxrtime'])
    
    curr_pts_admissions = admissions_df[admissions_df['subject_id'] == row['subject_id']]

    for hosp_index, hosp_row in curr_pts_admissions.iterrows():
        if hosp_row['admittime'] <= row['cxrtime'] <= hosp_row['dischtime']:
            cxr_embeddings_df.loc[index, 'hadm_id'] = hosp_row['hadm_id']
            cxr_embeddings_df.loc[index, 'hosp_time_delta'] = calc_time_delta_hrs(hosp_row['admittime'], row['cxrtime'])

100%|██████████| 377110/377110 [13:40<00:00, 459.85it/s]


In [9]:
cxr_embeddings_df[cxr_embeddings_df['icu_time_delta'].notnull()]

,dicom_id,subject_id,study_id,PerformedProcedureStepDescription,ViewPosition,ProcedureCodeSequence_CodeMeaning,ViewCodeSequence_CodeMeaning,PatientOrientationCodeSequence_CodeMeaning,densefeatures,predictions,cxrtime,hadm_id,stay_id,icu_time_delta,hosp_time_delta
71,469d0d94-3dad5068-efac76ef-a28cc502-68fe6275,10001884,50376803,CHEST (PORTABLE AP),AP,CHEST (PORTABLE AP),antero-posterior,Erect,"[0.0, 0.08129218, 0.10673142, 0.0032568781, 0....","[0.77309644, 0.5767501, 0.5, 0.3460556, 0.6746...",2131-01-15 04:45:09,26184834,37510196,96.417778,176.1025
72,7b25b3ed-e780a527-319cb7b3-02d5d071-f1cddee9,10001884,50712381,CHEST (PORTABLE AP),AP,CHEST (PORTABLE AP),antero-posterior,Erect,"[0.004358916, 0.10861397, 0.065632224, 0.00776...","[0.83478624, 0.6821995, 0.5, 0.50156933, 0.761...",2131-01-12 04:56:56,26184834,37510196,24.614167,104.298889
103,c1ad3e27-62d05ef8-95018fe3-b8bcfe4b-bbba0e1f,10001884,56722923,CHEST (PORTABLE AP),AP,CHEST (PORTABLE AP),antero-posterior,Erect,"[0.0028101602, 0.039716337, 0.16085272, 0.0050...","[0.75714236, 0.49764958, 0.5, 0.22080477, 0.76...",2131-01-13 04:49:18,26184834,37510196,48.486944,128.171667
119,9b1a8a51-2b8e4a04-1719059d-aa6bc888-7ace612b,10001884,59305618,CHEST (PORTABLE AP),AP,CHEST (PORTABLE AP),antero-posterior,Erect,"[0.007000305, 0.07577713, 0.0774089, 0.0110305...","[0.8755197, 0.5966458, 0.5, 0.18239394, 0.8151...",2131-01-14 10:34:28,26184834,37510196,78.239722,157.924444
147,e8c44648-ff02beea-3d5ff638-dec79b01-7df71a69,10002428,50027225,CHEST (PORTABLE AP),AP,CHEST (PORTABLE AP),antero-posterior,NaN,"[0.0, 0.008018429, 0.27646878, 0.023079809, 0....","[0.7306886, 0.71370304, 0.5, 0.1946802, 0.7639...",2156-04-16 03:00:29,28662225,33987268,82.603056,84.741389
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
377096,f7e95a22-cb958055-47114ddf-38532ef4-b4c172d5,19999287,52519175,CHEST (PORTABLE AP),AP,CHEST (PORTABLE AP),antero-posterior,Erect,"[0.0022679095, 0.06585629, 0.09802915, 0.00699...","[0.71268046, 0.5833337, 0.5, 0.25896987, 0.702...",2197-08-07 09:06:37,20175828,35165301,81.076944,84.143611
377100,2eb70dfe-52fa728e-a36e09be-ec0ed3cf-0a2ea7f0,19999287,58938059,CHEST (PORTABLE AP),AP,CHEST (PORTABLE AP),antero-posterior,Erect,"[0.001046607, 0.2166371, 0.19466288, 0.0, 0.36...","[0.9720168, 0.97115093, 0.5, 0.51177543, 0.917...",2197-08-05 09:37:46,20175828,35165301,33.596111,36.662778
377103,16b6c70f-6d36bd77-89d2fef4-9c4b8b0a-79c69135,19999442,58708861,CHEST (PORTABLE AP),AP,CHEST (PORTABLE AP),antero-posterior,Erect,"[0.0007100155, 0.001472501, 0.12827912, 0.0028...","[0.5088067, 0.14527085, 0.5, 0.10954579, 0.188...",2148-11-19 22:47:03,26785317,32336619,8.388889,12.784167
377107,58766883-376a15ce-3b323a28-6af950a0-16b793bd,19999987,55368167,CHEST (PORTABLE AP),AP,CHEST (PORTABLE AP),antero-posterior,Erect,"[0.0013912198, 0.018381726, 0.15159473, 0.0019...","[0.5444062, 0.098625995, 0.5, 0.09005868, 0.55...",2145-11-04 05:14:48,23865745,36195440,30.263333,31.613333


In [15]:
cxr_embeddings_df.to_pickle(os.path.join(preprocessing_dir, "cxr_embeddings_icu.pkl"))

In [7]:
cxr_embeddings_df['stay_id'].isna().mean()

0.6118373949245578

In [5]:
cxr_embeddings_df = pd.read_pickle(os.path.join(preprocessing_dir, "cxr_embeddings_stay.pkl"))